# Foreign Whispers — Colab GPU Backend

**Instructions:**
1. **Runtime > Change runtime type → T4 GPU**
2. Run **Cell 1 (Setup)** — wait for it to finish (~3 min)
3. Run **Cell 2 (Server)** — copy the `API_URL` to your Mac's `.env`

In [ ]:
# ==========================================
# CELL 1: SETUP (run once per session)
# ==========================================

import os

# Clone or update repo
if not os.path.isdir('foreign-whispers'):
    !git clone https://github.com/tilak30/foreign-whispers.git
else:
    !git -C foreign-whispers pull --rebase

%cd foreign-whispers

# Install tooling
!pip install uv pyngrok nest-asyncio -q

# Install project + rubberband (needed for pyrubberband)
!apt-get install -y rubberband-cli -q

# Install project deps (uses uv.lock for reproducibility)
!uv sync --no-dev

print('\n✅ Setup complete — run Cell 2 to start the server')

In [ ]:
# ==========================================
# CELL 2: START SERVER + NGROK TUNNEL
# ==========================================

import os, subprocess, time, socket
from pyngrok import ngrok
import nest_asyncio
nest_asyncio.apply()

os.environ['MPLBACKEND'] = 'Agg'

# --- PASTE YOUR NGROK TOKEN HERE ---
NGROK_TOKEN = '3D41N3dzj7hCpAfYDIXk2gdtNV1_3gBeNENvwHE32jporvsLR'
# -----------------------------------

PORT = 8080

# Kill anything already on the port
!fuser -k {PORT}/tcp 2>/dev/null || true
time.sleep(1)

# Kill any existing ngrok tunnels
ngrok.kill()
time.sleep(1)

# Start uvicorn in the background
log_file = open('/tmp/uvicorn.log', 'w')
server = subprocess.Popen(
    ['uv', 'run', 'uvicorn', 'api.src.main:app',
     '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=log_file,
    stderr=log_file,
)

# Wait for the server to actually be listening
print(f'Waiting for uvicorn to bind on port {PORT}...', end='')
for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            print(' ready!')
            break
    except OSError:
        print('.', end='', flush=True)
        time.sleep(2)
else:
    print(' TIMED OUT — check /tmp/uvicorn.log')
    raise RuntimeError('Server failed to start')

# Now open the ngrok tunnel AFTER the server is up
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print()
print('=' * 60)
print('✅ YOUR COLAB GPU BACKEND IS LIVE!')
print()
print(f'  API_URL={public_url}')
print()
print('Paste that line into your Mac .env file, then run:')
print('  docker compose --profile cpu up -d')
print('=' * 60)
print()
print('Server log → /tmp/uvicorn.log')
print('This cell will block. Keep it running while you use the app.')

# Block so the cell stays alive (runtime won't disconnect)
try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()
    ngrok.kill()
    print('\nServer stopped.')

In [ ]:
# ==========================================
# CELL 3 (optional): View server logs
# ==========================================
!tail -50 /tmp/uvicorn.log